<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione3/Lezione3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 3 — Conversazione & Memoria

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Martedì 26/05/2026

---

### 🎯 Obiettivi
- ✅ Gestire la conversation history (multi-turno)
- ✅ Implementare troncamento e sliding window
- ✅ Aggiungere lo streaming delle risposte
- ✅ Salvare e ricaricare la memoria su file JSON

In [ ]:
# Setup
!pip install anthropic -q
from google.colab import userdata
import anthropic, os, json

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 11.6 MB/s eta 0:00:00
✅ Setup completato!


---
## 1. Conversation History — il chatbot che ricorda

Il modello **non ha memoria propria**. Per farlo 'ricordare', dobbiamo inviargli tutta la conversazione ad ogni chiamata.

In [ ]:
# Chatbot multi-turno base
history = []

def chat(messaggio, system=None):
    """Invia un messaggio mantenendo la history."""
    # 1. Aggiungi il messaggio dell'utente
    history.append({"role": "user", "content": messaggio})

    # 2. Invia TUTTA la history al modello
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 500,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text

    # 3. Aggiungi la risposta alla history
    history.append({"role": "assistant", "content": testo})

    return testo

print("✅ Funzione chat pronta!")

✅ Funzione chat pronta!


In [ ]:
# Proviamo la memoria!
history = []  # Reset

print("👤 Mi chiamo Marco e sono di Sassari.")
r1 = chat("Mi chiamo Marco e sono di Sassari.")
print(f"🤖 {r1}\n")

print("👤 Qual è la capitale della Sardegna?")
r2 = chat("Qual è la capitale della Sardegna?")
print(f"🤖 {r2}\n")

print("👤 Come mi chiamo?")
r3 = chat("Come mi chiamo?")  # Ricorda il nome?
print(f"🤖 {r3}\n")

print(f"📊 Messaggi in history: {len(history)}")

👤 Mi chiamo Marco e sono di Sassari.
🤖 Piacere di conoscerti, Marco! Sassari è una bellissima città della Sardegna, con una ricca storia e cultura. 

C'è qualcosa di specifico su cui vorresti chiacchierare o su cui posso aiutarti? Sono qui per rispondere a domande, discutere di argomenti che ti interessano, o semplicemente fare conversazione.

👤 Qual è la capitale della Sardegna?
🤖 La capitale della Sardegna è **Cagliari**. 

È la città più grande dell'isola e si trova nel sud della Sardegna, affacciata sul Golfo degli Angeli. Sassari, dove vivi tu, è invece la seconda città più grande dell'isola ed è situata nel nord della Sardegna.

👤 Come mi chiamo?
🤖 Ti chiami **Marco**, come mi hai detto all'inizio della nostra conversazione! 😊

📊 Messaggi in history: 6


In [ ]:
# Vediamo quanti token stiamo usando
from anthropic import Anthropic

count = client.messages.count_tokens(
    model="claude-haiku-4-5-20251001",
    messages=history
)
print(f"📊 Token nella history attuale: {count.input_tokens}")
print(f"💰 Costo stimato prossima chiamata: ${count.input_tokens / 1_000_000 * 1.0:.6f}")
print()
print("💡 Nota: la history cresce ad ogni messaggio — dobbiamo gestirla!")

📊 Token nella history attuale: 248
💰 Costo stimato prossima chiamata: $0.000248

💡 Nota: la history cresce ad ogni messaggio — dobbiamo gestirla!


---
## 2. Gestire la Context Window

Tre strategie per evitare che la history cresca all'infinito.

In [ ]:
# STRATEGIA 1: Truncation
MAX_MESSAGGI = 6  # massimo 3 turni (user + assistant)

def chat_con_troncamento(messaggio, system=None):
    history.append({"role": "user", "content": messaggio})

    # Tronca se troppo lunga
    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]
        print(f"  ✂️  History troncata a {MAX_MESSAGGI} messaggi")

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=history
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

# Test
history = []
for i in range(5):
    r = chat_con_troncamento(f"Messaggio numero {i+1}. Quanti messaggi ricordi?")
    print(f"Turno {i+1} | History: {len(history)} msg | Risposta: {r[:80]}...")

Turno 1 | History: 2 msg | Risposta: Ricordo questo messaggio che mi stai inviando adesso. È il primo della nostra co...
Turno 2 | History: 4 msg | Risposta: Ricordo **2 messaggi**:

1. Il primo dove mi chiedevi quanti messaggi ricordo
2....
Turno 3 | History: 6 msg | Risposta: Ricordo **3 messaggi**:

1. Il primo dove mi chiedevi quanti messaggi ricordo
2....
  ✂️  History troncata a 6 messaggi
Turno 4 | History: 7 msg | Risposta: Ricordo **4 messaggi**:

1. Il primo dove mi chiedevi quanti messaggi ricordo
2....
  ✂️  History troncata a 6 messaggi
Turno 5 | History: 7 msg | Risposta: Ricordo **5 messaggi**:

1. Il primo dove mi chiedevi quanti messaggi ricordo
2....


In [ ]:
# STREAMING — output token per token
print("🌊 Risposta in streaming:\n")

full_text = ""
with client.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{"role": "user", "content": "Raccontami in 3 frasi cos'è il machine learning."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
        full_text += text

print(f"\n\n📊 Testo totale: {len(full_text)} caratteri")

🌊 Risposta in streaming:

# Machine Learning

Il machine learning è una branca dell'intelligenza artificiale che permette ai computer di imparare dai dati senza essere esplicitamente programmati per ogni singolo compito. L'algoritmo riconosce schemi e regolarità all'interno di grandi quantità di informazioni, migliorando le sue prestazioni con l'esperienza. Grazie a questa capacità, il machine learning è alla base di applicazioni come i sistemi di raccomandazione, il riconoscimento facciale e le auto autonome.

📊 Testo totale: 489 caratteri


---
## 3. Memoria Persistente su JSON

Salviamo la history su file — il chatbot ricorda tra sessioni diverse.

In [ ]:
import json, os

MEMORY_FILE = "chat_history.json"

def carica_storia():
    """Carica la history dal file JSON. Restituisce lista vuota se non esiste."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            storia = json.load(f)
            print(f"📂 Storia caricata: {len(storia)} messaggi precedenti")
            return storia
    print("🆕 Nessuna storia precedente — nuova conversazione")
    return []

def salva_storia(history):
    """Salva la history su file JSON."""
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"💾 Storia salvata: {len(history)} messaggi")

def chat_persistente(messaggio, system=None):
    """Chatbot con memoria persistente tra sessioni."""
    history.append({"role": "user", "content": messaggio})

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 400,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})

    salva_storia(history)  # Salva dopo ogni messaggio
    return testo

print("✅ Funzioni di persistenza pronte!")

✅ Funzioni di persistenza pronte!


In [ ]:
# Simulazione sessione 1
print("=" * 50)
print("SESSIONE 1")
print("=" * 50)

history = carica_storia()  # Carica eventuali messaggi precedenti

r = chat_persistente("Ciao! Mi chiamo Luca e studio AI engineering a Sassari.")
print(f"🤖 {r}\n")

r = chat_persistente("Qual è la lezione più difficile secondo te?")
print(f"🤖 {r}\n")

print("\n--- Fine sessione 1 ---")
print(f"History salvata con {len(history)} messaggi")

SESSIONE 1
🆕 Nessuna storia precedente — nuova conversazione
💾 Storia salvata: 2 messaggi
🤖 Ciao Luca! Piacere di conoscerti! 👋

Che bello che studi AI engineering a Sassari! È un campo affascinante e in grande espansione. 

Se hai domande su AI, machine learning, progetti di studio o qualsiasi altra cosa su cui posso aiutarti, sono tutto orecchi. Che si tratti di:
- Concetti teorici
- Help con coding
- Consigli su risorse di apprendimento
- Discussioni su argomenti tecnici

Dimmi pure come posso esserti utile! 😊

💾 Storia salvata: 4 messaggi
🤖 Bella domanda! Secondo me, le lezioni più impegnative in AI engineering tendono a essere:

**1. Matematica sottostante** - Algebra lineare, calcolo, probabilità e statistica. Non è tanto che sia "difficile" in sé, ma capire davvero *perché* funzionano gli algoritmi richiede una base solida.

**2. Intuizione vs. teoria** - Passare da "so come usare PyTorch" a "capisco veramente cosa sta succedendo dentro una rete neurale" è un salto enorme. Le fo

In [ ]:
# Simulazione sessione 2 — ricarica la memoria!
print("=" * 50)
print("SESSIONE 2 (nuova sessione, stessa memoria)")
print("=" * 50)

history = carica_storia()  # Ricarica dal file

r = chat_persistente("Come mi chiamo? Ricordi cosa stavo studiando?")
print(f"🤖 {r}")

SESSIONE 2 (nuova sessione, stessa memoria)
📂 Storia caricata: 4 messaggi precedenti
💾 Storia salvata: 6 messaggi
🤖 Sì, mi ricordo! 😊

Ti chiami **Luca** e studi **AI engineering a Sassari**.

Anche se devo essere onesto: in questa conversazione io mantengo la memoria solo all'interno di questo chat. Se chiudiamo e riapriamo una nuova conversazione, non me lo ricorderò più. È una limitazione di come funziono attualmente.

Ma in *questa* conversazione, sì, ho tutto quello che mi hai detto! 👍


---
## ⭐ Esercizi

In [ ]:
NOME_STUDENTE = "Anna"  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

✅ Notebook di: Anna


### Esercizio 1 — Chatbot multi-turno base ★☆☆
Crea un chatbot con system prompt WiData che mantiene la history. Fai almeno 4 domande collegate e verifica che risponda in modo coerente con i messaggi precedenti.

In [ ]:
# ESERCIZIO 1
history = []
system_widata = """ Sei l'assistente di WiData Srl, startup IoT in Sardegna.
Collega sempre le spiegazioni ai casi d'uso IoT e smart cities.
Sii conciso e orientato al business                               # ← scrivi il system prompt WiData
"""

domanda1 = chat("Cosa fa WiData Srl?", system=system_widata)
print(f"🤖 {domanda1}")
domanda2 = chat("Quali sono i casi d'uso di WiData Srl?", system=system_widata)
print(f"🤖 {domanda2}")
domanda3 = chat("Come raccogliete e analizzate i dati dai sensori?", system=system_widata)
print(f"🤖 {domanda3}")
domanda4 = chat("Perchè lavorate solo in Sardegna?", system=system_widata)
print(f"🤖 {domanda4}")

🤖 # WiData Srl - Smart IoT Solutions per le Città Intelligenti

WiData è una startup IoT sarda specializzata nello sviluppo di **soluzioni di connettività e raccolta dati intelligenti** per smart cities e applicazioni industriali.

## Aree principali:

🔌 **IoT & Sensori**
- Piattaforme di monitoraggio in tempo reale
- Gestione di reti di dispositivi distribuiti

📊 **Raccolta e Analisi Dati**
- Dashboard e analytics per insights actionable
- Ottimizzazione dei processi urbani

🌍 **Smart City Solutions**
- Gestione efficiente di risorse (acqua, energia, mobilità)
- Monitoraggio ambientale e qualità dell'aria
- Sicurezza e servizi pubblici

💼 **Applicazioni B2B**
- Supporto ad amministrazioni pubbliche
- Integrazione con infrastrutture esistenti

## Valore aggiunto:
✅ Know-how locale sardo + competenze IoT globali
✅ Focus sulla sostenibilità e efficienza
✅ Soluzioni scalabili e adattabili

Posso aiutarti su aspetti specifici: tecnologie utilizzate, casi d'uso concreti o come WiData affron

### Esercizio 2 — Sliding Window ★★☆
Implementa la sliding window: mantieni sempre il system prompt + gli ultimi 4 turni (8 messaggi). Testa che dopo 6 turni il chatbot non ricordi più i primi messaggi ma ricordi gli ultimi.

In [ ]:
# ESERCIZIO 2
MAX_TURNS = 4

def chat_sliding_window(messaggio, system=None):
    history.append({"role": "user", "content": messaggio})

    window= history[-MAX_TURNS*2:]
    if system is not None:
        risposta = client.messages.create(model="claude-haiku-4-5-20251001",max_tokens=300,system=system,
        messages=window
        )
    else:
        risposta = client.messages.create(model="claude-haiku-4-5-20251001",max_tokens=300,
            messages=window
        )

    risposta = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=300, messages=window)
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})

    return testo


# Test: la prima informazione viene dimenticata dopo 4 turni?
history = []
chat_sliding_window("Mi chiamo Anna")
chat_sliding_window("Abito a Castelsardo")
chat_sliding_window("Sono una studentessa")
chat_sliding_window("Mi piace il colore arancione")
chat_sliding_window("Ieri come era il mare?")
risposta = chat_sliding_window("Come mi chiamo?")
print(f"🤖 {risposta}")

#L'informazione sul mio nome viene dimenticata

🤖 Non lo so! 😄

Non mi hai detto il tuo nome. Posso solo sapere quello che mi racconti nella nostra conversazione.

Mi hai detto che:
- Sei una studentessa
- Sei (probabilmente) da Castelsardo
- Ti piace il colore arancione

Ma il tuo nome... rimane un mistero! 🙂

Se vuoi, puoi dirmi come ti chiami! Altrimenti posso semplicemente chiamarti "studentessa" o "tu". Come preferisci?


### Esercizio 3 — Chatbot con streaming ★★☆
Riscrivi la funzione `chat()` usando lo streaming. L'output deve apparire parola per parola nel terminale. Aggiungi anche il conteggio dei token al termine.

In [ ]:
 # ESERCIZIO 3
def chat_streaming(messaggio, history, system=None):
    history.append({"role": "user", "content": messaggio})
    full_text = ""

    with client.messages.stream(model="claude-haiku-4-5-20251001", max_tokens=300,messages=history) as stream:
        for text in stream.text_stream:
            print(text, end='', flush=True)
            full_text += text


    finale = stream.get_final_message()
    history.append({"role": "assistant", "content": full_text})
    print(f"\n Token — input: {finale.usage.input_tokens}, output: {finale.usage.output_tokens}")

# Test
history = []
chat_streaming("Spiegami RAG in 3 frasi", history)


# RAG (Retrieval-Augmented Generation)

1. **RAG è una tecnica** che combina il recupero di informazioni da documenti esterni con la generazione di testo da parte di un modello AI.

2. **Il funzionamento**: il sistema cerca prima i documenti rilevanti nel database, poi usa questi risultati come contesto per generare risposte più accurate e aggiornate.

3. **Il vantaggio principale**: consente ai modelli AI di fornire risposte basate su fonti affidabili senza dover "imparare" ogni fatto nuovo, riducendo allucinazioni e permettendo di usare dati aggiornati.
 Token — input: 19, output: 163


### Esercizio 4 — Chatbot con memoria persistente ★★★ (Deliverable!)

Costruisci il chatbot completo `chatbot_cli.py` con:
- History multi-turno
- Sliding window (max 10 messaggi)
- Streaming
- Memoria su JSON persistente
- System prompt WiData
- Loop interattivo con `input()` (digita 'esci' per uscire)

In [ ]:
# ESERCIZIO 4 — Chatbot completo (DELIVERABLE)
# Scrivi qui il codice completo del chatbot

import anthropic, json, os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

MEMORY_FILE = "chatbot_widata.json"
MAX_MESSAGGI = 10

SYSTEM = """ Sei un assistente di widata, in particolare ti occupi del supporto clienti
Devi rispondere solo su domande su prodotti IoT
avere un tono professionale
per domande rigurdo ai prezzi rispondere di contattare al nostro consulente per avere delle offerte personalizzate
devi rifiutare qualsiasi domanda fuori tema rispondendo con il tuo ruolo
                                                                                                                    # ← Scrivi qui il tuo system prompt WiData
"""

def carica_storia():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            storia = json.load(f)
            print(f"📂 Storia caricata: {len(storia)} messaggi precedenti")
            return storia
    print("🆕 Nessuna storia precedente — nuova conversazione")
    return []

def salva_storia(history):
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"💾 Storia salvata: {len(history)} messaggi")

def chat(messaggio, history,SYSTEM=None):
    history.append({"role": "user", "content": messaggio})

    full_text = ""

    with client.messages.stream(model="claude-haiku-4-5-20251001",max_tokens=400,messages=history[-MAX_MESSAGGI:],
        system=SYSTEM
    ) as stream:
        for text in stream.text_stream:
            print(text, end='', flush=True)
            full_text += text
        finale = stream.get_final_message()
        history.append({"role": "assistant", "content": full_text})
        print(f"\n Token — input: {finale.usage.input_tokens}, output: {finale.usage.output_tokens}")


    salva_storia(history)
    return full_text


def main():
    history = carica_storia()
    print("🤖 Chatbot WiData avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        risposta = chat(utente, history,SYSTEM)

main()

🆕 Nessuna storia precedente — nuova conversazione
🤖 Chatbot WiData avviato. Digita 'esci' per uscire.

Tu: Chi sei?
# Ciao! 👋

Sono l'assistente di supporto clienti **WiData**, specializzato in prodotti e soluzioni **IoT** (Internet of Things).

Sono qui per aiutarti con:
- Domande su prodotti e soluzioni IoT WiData
- Chiarimenti tecnici e funzionali
- Informazioni generali sui nostri servizi

**Per quanto riguarda i prezzi e le offerte personalizzate**, ti consiglio di contattare direttamente uno dei nostri consulenti, che saranno in grado di proporti soluzioni su misura per le tue esigenze.

Come posso assisterti oggi?
 Token — input: 131, output: 168
💾 Storia salvata: 2 messaggi
Tu: esci
👋 Arrivederci!


---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione3_TUONOME.ipynb`
4. Carica su GitHub in `lezione3/`

```bash
git add lezione3/
git commit -m "Lezione 3 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 28/05)
Leggi **Huyen Cap. 6 — sezione RAG**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*